# NB35: second seed of NB33 (same recipe, a new seed set per RUN letter)

Select **L4**, then **Run all**, with the Drive mounted. This is NB33 unchanged except the random seed: fold k uses seed 1000n+42+k, n = 1 for RUN b, 2 for c, 3 for d, in place of 42+k, and outputs go to `nb35_<RUN>_wide_interval` so nothing from NB33 is touched. Nothing here is scored or frozen; the ensemble of the two seeds is evaluated offline against NB33's frozen gate. Resumable: rerun all after a disconnect and finished folds skip.


In [ ]:
RUN='b'
from google.colab import drive
drive.mount('/content/drive')
import subprocess,sys,os
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
subprocess.run([sys.executable,'-m','pip','install','-q','chemprop==2.3.1','lightning==2.6.6'],check=True)


In [ ]:
from pathlib import Path
import json,hashlib,zipfile,shutil,time
import torch
assert torch.cuda.is_available(), 'Select an L4 GPU runtime first.'
print('GPU:',torch.cuda.get_device_name(0))
CACHE=Path('/content/nb33_cache');CACHE.mkdir(exist_ok=True)
PAYLOAD=Path('/content/drive/MyDrive/cyp_payloads/nb35_seed_ae4ef8ab97ae.zip')
assert PAYLOAD.exists(), 'Payload missing: mount your Google account Drive.'
assert hashlib.sha256(PAYLOAD.read_bytes()).hexdigest()=='ae4ef8ab97ae28b8e0e7c5785cadfe25f89f2fda1d5fb330c60030d74dd16476'
with zipfile.ZipFile(PAYLOAD) as z:
    for n in z.namelist():assert not Path(n).is_absolute() and '..' not in Path(n).parts
    z.extractall(CACHE)
manifest=json.loads((CACHE/'manifest.json').read_text())
for n,digest in manifest.items():assert hashlib.sha256((CACHE/n).read_bytes()).hexdigest()==digest,n
OUT=Path('/content/drive/MyDrive/cyp_results')/f'nb35_{RUN}_wide_interval';OUT.mkdir(parents=True,exist_ok=True)
for source in (CACHE/'inputs').iterdir():
    dest=OUT/source.name
    if dest.exists():assert dest.read_bytes()==source.read_bytes(), 'Changed inputs: choose a new RUN.'
    else:shutil.copy2(source,dest)
os.chdir(CACHE);os.environ.update(NB33_OUT=str(OUT),NB33_DEVICE='gpu',NB33_SEED=str(1000*(ord(RUN)-96)+42),OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
print('Data hashes verified; outputs:',OUT)
print(json.loads((OUT/'data_audit.json').read_text()))


In [ ]:
# Finite five-fold run. Each fold is a separate process to release GPU memory.
started=time.time()
for fold in range(5):
    print(f'\nFOLD {fold}/4: starting or resuming',flush=True)
    with (OUT/f'fold{fold}.log').open('a') as log:
        proc=subprocess.Popen([sys.executable,'-u','train_nb33.py','--fold',str(fold),'--batch-size','128'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            rc=proc.wait()
        except BaseException:
            proc.terminate();proc.wait();raise
    assert rc==0, f'Fold {fold} stopped. Read its log; rerun to resume from last saved epoch.'
    pass
print('All folds finished in',round((time.time()-started)/3600,2),'hours')


In [ ]:
RESULT_ZIP=OUT.parent/f'nb35_{RUN}_results.zip'
with zipfile.ZipFile(RESULT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ('.json','.npz','.log'):z.write(p,p.relative_to(OUT))
print('RESULTS SAVED:',RESULT_ZIP)
